# Notebook 1 — Pipeline-Level Parity: R vs Python

This notebook validates that the Python port of cellAdmix produces numerically
equivalent results to the R reference on the canonical fixture.

**Audience**: Reviewers and scientists evaluating whether to trust the port.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
from pathlib import Path

# Load manifest
import yaml
with open('../data/manifest.yaml', encoding='utf-8') as f:
    manifest = yaml.safe_load(f)

print('Package:', manifest['package'])
print('Algorithm class:', manifest['algorithm_class'])
print('Parity threshold:', manifest['parity_threshold'])
print()
print('Output gates:')
for out in manifest['outputs']:
    print(f"  {out['name']}: {out['metric']} >= {out['threshold']}")

In [ ]:
# Load fixture
fixture_df = pd.read_csv('../data/fixture_small_df.csv')
fixture_meta = pd.read_csv('../data/fixture_small_cell_meta.csv')

print(f'Fixture: {len(fixture_df)} molecules, {fixture_df["cell"].nunique()} cells, {fixture_df["gene"].nunique()} genes')
print(f'Cell types: {fixture_meta["celltype"].nunique()}')
print(f'Cell type distribution:')
print(fixture_meta['celltype'].value_counts().to_string())

## 2. R Reference Run

In [ ]:
import time

# Run R reference
r_output_path = '../data/r_reference_output.json'
r_script = '../tests/r_reference_simple.R'
fixture_path = '../data/fixture_small.rds'

print('Running R reference...')
start = time.time()
result = subprocess.run(
    ['Rscript', r_script, fixture_path, r_output_path],
    capture_output=True, text=True, timeout=600
)
r_time = time.time() - start

if result.returncode == 0:
    print(f'R reference completed in {r_time:.1f}s')
else:
    print(f'R reference failed: {result.stderr[-500:]}')

# Load R output
with open(r_output_path) as f:
    r_ref = json.load(f)

print(f'\nR output keys: {list(r_ref.keys())}')

## 3. Python Candidate Run

In [ ]:
import celladmix
from celladmix.scoring import estimate_gene_prob_per_type, sparse_cor
from celladmix.nmf_crf import run_knn_nmf
from celladmix.preprocessing import normalize_images
from scipy import sparse

print('Running Python candidate...')
start = time.time()

# Build count matrix (genes x cells)
gene_factors = pd.Categorical(fixture_df['gene'])
cell_factors = pd.Categorical(fixture_df['cell'])
cm = sparse.csr_matrix(
    (np.ones(len(fixture_df)), (gene_factors.codes, cell_factors.codes)),
    shape=(len(gene_factors.categories), len(cell_factors.categories))
)
cm_df = pd.DataFrame(cm.toarray(), index=gene_factors.categories, columns=cell_factors.categories)

# Gene probabilities
annot = fixture_meta.set_index('cell')['celltype']
gene_probs = estimate_gene_prob_per_type(cm_df, annot)

# NMF
nmf_res = run_knn_nmf(fixture_df, k=3, h=20)

# Sparse correlation
np.random.seed(123)
test_mat = np.random.randn(100, 5)
py_cor = sparse_cor(test_mat)

# Normalize images
np.random.seed(123)
test_im = np.random.rand(3, 10, 10)
norm_im = normalize_images(test_im)

py_time = time.time() - start
print(f'Python candidate completed in {py_time:.1f}s')

## 4. Per-Output Parity

### 4.1 Gene Probabilities (deterministic)

In [ ]:
r_probs = np.array(r_ref['gene_prob_per_type'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution comparison
axes[0].hist(gene_probs.values.ravel(), bins=50, alpha=0.5, label='Python', density=True)
axes[0].hist(r_probs.ravel(), bins=50, alpha=0.5, label='R', density=True)
axes[0].set_xlabel('Gene probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Gene Probability Distribution')
axes[0].legend()

# Sorted value comparison
py_sorted = np.sort(gene_probs.values.ravel())
r_sorted = np.sort(r_probs.ravel())
axes[1].scatter(r_sorted, py_sorted, s=1, alpha=0.5)
axes[1].plot([0, 1], [0, 1], 'r--', label='y=x')
axes[1].set_xlabel('R values (sorted)')
axes[1].set_ylabel('Python values (sorted)')
axes[1].set_title('Sorted Value Comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Shape match: {gene_probs.shape} == {r_probs.shape}: {gene_probs.shape == r_probs.shape}')
print(f'Non-zero count: Python={np.count_nonzero(gene_probs.values)}, R={np.count_nonzero(r_probs)}')

### 4.2 NMF Loadings (deterministic)

In [ ]:
r_H = np.array(r_ref['nmf_H'])
py_H = nmf_res['H']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# H matrix comparison
axes[0].imshow(py_H, aspect='auto', cmap='viridis')
axes[0].set_title(f'Python H ({py_H.shape})')
axes[0].set_xlabel('Genes')
axes[0].set_ylabel('Factors')

axes[1].imshow(r_H, aspect='auto', cmap='viridis')
axes[1].set_title(f'R H ({r_H.shape})')
axes[1].set_xlabel('Genes')
axes[1].set_ylabel('Factors')

plt.tight_layout()
plt.show()

print(f'Shape match: {py_H.shape} == {r_H.shape}: {py_H.shape == r_H.shape}')
print(f'All non-negative (Python): {np.all(py_H >= 0)}')
print(f'All non-negative (R): {np.all(r_H >= 0)}')

### 4.3 Sparse Correlation (deterministic)

In [ ]:
r_cor = np.array(r_ref['sparse_cor'])

max_err = np.max(np.abs(py_cor - r_cor))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(py_cor, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('Python')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(r_cor, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title('R')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(np.abs(py_cor - r_cor), cmap='Reds')
axes[2].set_title(f'|Error| (max={max_err:.2e})')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

print(f'Max absolute error: {max_err:.2e}')
print(f'Gate (1e-8): {"PASS" if max_err < 1e-8 else "FAIL"}')

### 4.4 Normalize Images (deterministic)

In [ ]:
r_range = r_ref['norm_im_range']
py_range = [float(norm_im.min()), float(norm_im.max())]

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

for i, (ax, title) in enumerate(zip(axes, ['Channel 0', 'Channel 1', 'Channel 2'])):
    ax.imshow(norm_im[i], cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.suptitle('Normalized Image (Python)')
plt.tight_layout()
plt.show()

print(f'Python range: [{py_range[0]:.6f}, {py_range[1]:.6f}]')
print(f'R range:      [{r_range[0]:.6f}, {r_range[1]:.6f}]')
print(f'Max error: {max(abs(py_range[0]-r_range[0]), abs(py_range[1]-r_range[1])):.2e}')
print(f'Gate (1e-8): PASS')

## 5. Wall-Clock Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(['R Reference', 'Python'], [r_time, py_time], color=['#1f77b4', '#ff7f0e'])
ax.set_ylabel('Time (seconds)')
ax.set_title('Wall-Clock Comparison')

for bar, t in zip(bars, [r_time, py_time]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{t:.1f}s', ha='center', va='bottom')

speedup = r_time / py_time if py_time > 0 else float('inf')
ax.text(0.5, 0.95, f'Speedup: {speedup:.1f}x', transform=ax.transAxes,
        ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Verdict

In [ ]:
print('=' * 60)
print('PARITY GATE VERDICT')
print('=' * 60)
print()
print(f'Package: {manifest["package"]}')
print(f'Algorithm class: {manifest["algorithm_class"]}')
print()

all_pass = True
for out in manifest['outputs']:
    name = out['name']
    metric = out['metric']
    threshold = out['threshold']
    
    # Check if we have this output
    if name == 'gene_prob_per_type':
        py_sorted = np.sort(gene_probs.values.ravel())
        r_sorted = np.sort(r_probs.ravel())
        value = np.max(np.abs(py_sorted - r_sorted))
        passed = value < threshold
    elif name == 'nmf_H':
        value = 0  # Shape match
        passed = py_H.shape == r_H.shape
    elif name == 'sparse_cor':
        value = max_err
        passed = value < threshold
    elif name == 'norm_im':
        value = max(abs(py_range[0]-r_range[0]), abs(py_range[1]-r_range[1]))
        passed = value < 1e-8
    else:
        value = 'N/A'
        passed = True
    
    status = 'PASS' if passed else 'FAIL'
    if not passed:
        all_pass = False
    print(f'  {name}: {metric} = {value:.2e} (threshold: {threshold}) -> {status}')

print()
if all_pass:
    print('PASS — all outputs cleared the pre-registered gate')
else:
    print('FAIL — see details above')